# Dudley Car Availability — Data Cleaning

## Purpose
Prepare the Census 2021 car or van availability dataset for Dudley
for later analysis of potential transport barriers to employment.

## Objectives
- Inspect the raw dataset and identify non-data rows.
- Rename columns clearly.
- Check missing values, duplicates and data types.
- Validate counts and Dudley LSOA coverage.
- Save a cleaned CSV for later analysis.

## Input
`raw_datasets/dudley_car_availability.csv`

## Output
`cleaned_datasets/dudley_car_availability_cleaned.csv`

## Ethical Considerations
This dataset describes households within neighbourhoods, not individuals.
Having no car or van does not necessarily mean someone cannot travel to work.
Public transport, walking and cycling may provide alternatives.

## Additional Comments
Keep the raw dataset unchanged. This notebook covers cleaning and
validation only; percentages and further analysis will be completed later.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

## 1. Inspect the Raw File

Inspect the first ten lines to identify the dataset information,
header row and beginning of the data before loading the CSV.

In [5]:
# Import pandas for the cleaning steps.
import pandas as pd

# Define the raw dataset path.
raw_path = "raw_datasets/dudley_car_availability.csv"

# Display the first ten lines, including any metadata and blank lines.
with open(raw_path, "r", encoding="utf-8-sig") as file:
    for line_number in range(1, 11):
        line = file.readline()
        print(f"{line_number}: {line.rstrip()}")

1: TS045 - Car or van availability,,,,,,
2: ONS Crown Copyright Reserved [from Nomis on 14 October 2026],,,,,,
3: Population :,All households,,,,,
4: Units      :,Households,,,,,
5: Date       :,2021,,,,,
6: ,,,,,,
7: 2021 super output area - lower layer,mnemonic,Total: All households,No cars or vans in household,1 car or van in household,2 cars or vans in household,3 or more cars or vans in household
8: ,,,,,,
9: Dudley 001A,E01009763,882,261,344,207,70
10: Dudley 001B,E01009764,632,76,264,215,77


### Findings

- The file contains Census 2021 table TS045: Car or van availability,
  accessed through Nomis on 14 October 2026.
- The population and units are households, not individual residents.
- Column headers appear on line 7, so the first six lines must be
  skipped when loading the CSV.
- Line 8 is blank, and the first data record appears on line 9.
- Seven columns contain the LSOA name, LSOA code, total households
  and four car or van availability categories.

---

## 2. Load and Preview the Dataset

Load the CSV, skipping the six lines before the column headers.
Check its dimensions and preview the first five rows.

In [6]:
# Load the CSV using the headers on line 7.
car_df = pd.read_csv(
    raw_path,
    skiprows=6,
    encoding="utf-8-sig"
)

# Display the number of rows and columns.
print(f"Rows: {car_df.shape[0]}")
print(f"Columns: {car_df.shape[1]}")

# Preview the first five rows.
car_df.head()

Rows: 211
Columns: 7


,2021 super output area - lower layer,mnemonic,Total: All households,No cars or vans in household,1 car or van in household,2 cars or vans in household,3 or more cars or vans in household
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dudley 001A,E01009763,882.0,261.0,344.0,207.0,70.0
2,Dudley 001B,E01009764,632.0,76.0,264.0,215.0,77.0
3,Dudley 001C,E01009767,658.0,152.0,274.0,173.0,59.0
4,Dudley 001D,E01009769,824.0,336.0,318.0,121.0,49.0


### Findings

- The dataset loaded with 211 rows and 7 columns.
- The first row is completely empty.
- The preview shows Dudley LSOA names, codes and household counts.
- Counts currently display with decimal places; their data types will
  be checked before conversion to integers.
- Empty rows and any other non-data records need inspection before removal.

---

## 3. Check Completely Empty Rows

Identify rows where every column is missing and inspect them before removal.

In [7]:
# Identify rows where all values are missing.
empty_rows = car_df.isna().all(axis=1)

# Count and display the completely empty rows.
print(f"Completely empty rows: {empty_rows.sum()}")
car_df.loc[empty_rows]

Completely empty rows: 4


,2021 super output area - lower layer,mnemonic,Total: All households,No cars or vans in household,1 car or van in household,2 cars or vans in household,3 or more cars or vans in household
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Findings

- Four completely empty rows were identified at indices 0, 204, 205 and 206.
- These rows contain no LSOA information or household counts.
- They can be removed during cleaning. Other non-data records will
  be checked first.

---

## 4. Inspect Other Non-Data Records

Check for non-empty rows whose mnemonic does not match the expected
code format: E followed by eight digits. Inspect these records before
deciding whether to remove them.

In [8]:
# Developed with ChatGPT assistance to inspect potential non-data records.

# Identify codes matching the expected format.
valid_code_format = (
    car_df["mnemonic"]
    .astype("string")
    .str.fullmatch(r"E\d{8}", na=False)
)

# Exclude completely empty rows to inspect other unmatched records.
other_non_data = car_df.loc[~valid_code_format & ~empty_rows]

# Count and display these records.
print(f"Other records requiring inspection: {len(other_non_data)}")
other_non_data

Other records requiring inspection: 4


,2021 super output area - lower layer,mnemonic,Total: All households,No cars or vans in household,1 car or van in household,2 cars or vans in household,3 or more cars or vans in household
207,NaN,In order to protect against disclosure of pers...,NaN,NaN,NaN,NaN,NaN
208,NaN,have been swapped between different geographic...,NaN,NaN,NaN,NaN,NaN
209,NaN,by small amounts. Small counts at the lowest g...,NaN,NaN,NaN,NaN,NaN
210,NaN,affected.,NaN,NaN,NaN,NaN,NaN


### Findings

- Four additional non-data rows were identified at indices 207–210.
- These contain disclosure-control notes in the mnemonic column,
  with no LSOA names or household counts.
- Together with the four empty rows, eight records can be excluded
  from the cleaned dataset.
- The disclosure-control notes will remain preserved in the unchanged
  raw CSV.

---

## 5. Remove Non-Data Records

Create a separate cleaned dataframe containing rows with the expected
code format. This excludes the four empty rows and four disclosure-note
rows while keeping the original dataframe unchanged.

Reset the index after filtering.

In [9]:
# Developed with ChatGPT assistance to remove verified non-data records.

# Keep rows with correctly formatted codes in a separate dataframe.
car_clean = (
    car_df.loc[valid_code_format]
    .copy()
    .reset_index(drop=True)
)

# Report the number of removed and remaining rows.
print(f"Rows removed: {len(car_df) - len(car_clean)}")
print(f"Rows remaining: {len(car_clean)}")
print(f"Columns: {car_clean.shape[1]}")

# Preview the first five retained records.
car_clean.head()

Rows removed: 8
Rows remaining: 203
Columns: 7


,2021 super output area - lower layer,mnemonic,Total: All households,No cars or vans in household,1 car or van in household,2 cars or vans in household,3 or more cars or vans in household
0,Dudley 001A,E01009763,882.0,261.0,344.0,207.0,70.0
1,Dudley 001B,E01009764,632.0,76.0,264.0,215.0,77.0
2,Dudley 001C,E01009767,658.0,152.0,274.0,173.0,59.0
3,Dudley 001D,E01009769,824.0,336.0,318.0,121.0,49.0
4,Dudley 001E,E01009770,637.0,152.0,274.0,157.0,54.0


### Findings

- Eight non-data rows were removed: four empty rows and four
  disclosure-note rows.
- The cleaned dataframe contains 203 rows and 7 columns.
- The index was reset, and the preview begins with Dudley 001A.
- The original dataframe and raw CSV remain unchanged.
- LSOA coverage, duplicates and household counts still require validation.

---

## 6. Rename Columns

Replace the original column headings with concise names using underscores.
Keep household counts clearly labelled and use consistent LSOA identifiers
for later comparisons with the other datasets.

In [10]:
# Developed with ChatGPT assistance to create clear column names.

# Map the original headings to descriptive names.
car_clean = car_clean.rename(columns={
    "2021 super output area - lower layer": "LSOA_Name",
    "mnemonic": "LSOA_Code",
    "Total: All households": "Total_Households",
    "No cars or vans in household": "No_Car_Van_Households",
    "1 car or van in household": "One_Car_Van_Households",
    "2 cars or vans in household": "Two_Cars_Vans_Households",
    "3 or more cars or vans in household": "Three_Plus_Cars_Vans_Households"
})

# Display the updated column names and first five rows.
print(car_clean.columns.tolist())
car_clean.head()

['LSOA_Name', 'LSOA_Code', 'Total_Households', 'No_Car_Van_Households', 'One_Car_Van_Households', 'Two_Cars_Vans_Households', 'Three_Plus_Cars_Vans_Households']


,LSOA_Name,LSOA_Code,Total_Households,No_Car_Van_Households,One_Car_Van_Households,Two_Cars_Vans_Households,Three_Plus_Cars_Vans_Households
0,Dudley 001A,E01009763,882.0,261.0,344.0,207.0,70.0
1,Dudley 001B,E01009764,632.0,76.0,264.0,215.0,77.0
2,Dudley 001C,E01009767,658.0,152.0,274.0,173.0,59.0
3,Dudley 001D,E01009769,824.0,336.0,318.0,121.0,49.0
4,Dudley 001E,E01009770,637.0,152.0,274.0,157.0,54.0


### Findings

- All seven columns have been renamed successfully.
- LSOA_Name and LSOA_Code now follow the naming convention used
  in the other cleaned datasets.
- The five count columns clearly identify households and their
  car or van availability categories.
- Renaming changed only the headings; the data values were preserved.

---

## 7. Check Missing Values and Duplicates

Check each column for missing values, then identify duplicate rows
and repeated LSOA codes. Each LSOA should appear only once.

In [11]:
# Developed with ChatGPT assistance to check missing values and duplicates.

# Count missing values in each column.
print("Missing values per column:")
print(car_clean.isna().sum())

# Count fully duplicated rows and repeated LSOA codes.
print(f"\nDuplicate rows: {car_clean.duplicated().sum()}")
print(f"Duplicate LSOA codes: {car_clean['LSOA_Code'].duplicated().sum()}")

# Report the number of unique LSOA codes.
print(f"Unique LSOA codes: {car_clean['LSOA_Code'].nunique()}")

Missing values per column:
LSOA_Name                          0
LSOA_Code                          0
Total_Households                   0
No_Car_Van_Households              0
One_Car_Van_Households             0
Two_Cars_Vans_Households           0
Three_Plus_Cars_Vans_Households    0
dtype: int64

Duplicate rows: 0
Duplicate LSOA codes: 0
Unique LSOA codes: 203


### Findings

- No missing values were found in any of the seven columns.
- No duplicate rows or duplicate LSOA codes were found.
- There are 203 unique LSOA codes, with one row per code.
- No missing-value treatment or duplicate removal is required.

---

## 8. Validate Dudley LSOA Coverage

Check that all area names follow the Dudley LSOA naming format.
Compare LSOA codes with the cleaned economic activity dataset to
confirm that both datasets cover the same areas.

In [12]:
# Developed with ChatGPT assistance to validate Dudley LSOA coverage.

# Check that each area name follows the expected Dudley format.
valid_names = (
    car_clean["LSOA_Name"]
    .astype("string")
    .str.fullmatch(r"Dudley \d{3}[A-Z]", na=False)
)
print(f"Names not matching Dudley format: {(~valid_names).sum()}")

# Load the cleaned economic activity dataset for comparison.
economic_df = pd.read_csv(
    "cleaned_datasets/dudley_economic_activity_cleaned.csv"
)

# Compare the unique LSOA codes in both datasets.
car_codes = set(car_clean["LSOA_Code"])
economic_codes = set(economic_df["LSOA_Code"])

print(f"Car availability LSOAs: {len(car_codes)}")
print(f"Economic activity LSOAs: {len(economic_codes)}")
print("Only in car availability:", sorted(car_codes - economic_codes))
print("Only in economic activity:", sorted(economic_codes - car_codes))

Names not matching Dudley format: 0
Car availability LSOAs: 203
Economic activity LSOAs: 203
Only in car availability: []
Only in economic activity: []


### Findings

- All area names match the expected Dudley LSOA naming format.
- Both datasets contain 203 unique LSOA codes.
- No codes occur exclusively in either dataset, confirming identical
  LSOA coverage.
- The datasets can later be joined using LSOA_Code. Their population
  units differ: car availability measures households, while economic
  activity measures people.

---

## 9. Validate Household Counts

Check that the five household count columns contain numeric,
non-negative whole numbers before converting them to integers.

In [13]:
# Developed with ChatGPT assistance to validate household counts.

# Define the columns containing household counts.
count_columns = [
    "Total_Households",
    "No_Car_Van_Households",
    "One_Car_Van_Households",
    "Two_Cars_Vans_Households",
    "Three_Plus_Cars_Vans_Households"
]

# Create a numeric copy for validation without changing the cleaned data.
numeric_counts = car_clean[count_columns].apply(
    pd.to_numeric, errors="coerce"
)

# Summarise invalid values in each count column.
count_checks = pd.DataFrame({
    "Missing_or_Non_Numeric": numeric_counts.isna().sum(),
    "Negative": numeric_counts.lt(0).sum(),
    "Fractional": (
        numeric_counts.notna() & numeric_counts.mod(1).ne(0)
    ).sum()
})

count_checks

,Missing_or_Non_Numeric,Negative,Fractional
Total_Households,0,0,0
No_Car_Van_Households,0,0,0
One_Car_Van_Households,0,0,0
Two_Cars_Vans_Households,0,0,0
Three_Plus_Cars_Vans_Households,0,0,0


### Findings

- All five household count columns contain numeric values with no
  missing entries.
- No negative or fractional values were found.
- The counts can be converted to integers without rounding or
  changing their values.

---

## 10. Convert Household Counts to Integers

Convert the five validated count columns to integers so that household
counts display as whole numbers. No rounding is required.

In [14]:
# Developed with ChatGPT assistance to convert validated counts to integers.

# Convert the five household count columns without rounding.
car_clean[count_columns] = numeric_counts.astype("int64")

# Confirm the updated data types.
print(car_clean[count_columns].dtypes)

# Preview the first five rows.
car_clean.head()

Total_Households                   int64
No_Car_Van_Households              int64
One_Car_Van_Households             int64
Two_Cars_Vans_Households           int64
Three_Plus_Cars_Vans_Households    int64
dtype: object


,LSOA_Name,LSOA_Code,Total_Households,No_Car_Van_Households,One_Car_Van_Households,Two_Cars_Vans_Households,Three_Plus_Cars_Vans_Households
0,Dudley 001A,E01009763,882,261,344,207,70
1,Dudley 001B,E01009764,632,76,264,215,77
2,Dudley 001C,E01009767,658,152,274,173,59
3,Dudley 001D,E01009769,824,336,318,121,49
4,Dudley 001E,E01009770,637,152,274,157,54


### Findings

- All five household count columns now use the int64 data type.
- Counts display as whole numbers, with no rounding or changes
  to their values.
- LSOA names and codes remain unchanged.

---

## 11. Check Household Totals

Check whether the four car or van availability categories sum to
Total_Households for each LSOA. Inspect any differences without
changing the source counts.

In [15]:
# Developed with ChatGPT assistance to check household total consistency.

# Sum the four availability categories for each LSOA.
category_totals = car_clean[count_columns[1:]].sum(axis=1)

# Identify rows where the category sum differs from the reported total.
total_difference = category_totals - car_clean["Total_Households"]
mismatched_totals = total_difference.ne(0)

# Report the number of mismatches.
print(f"Rows where categories do not match the total: {mismatched_totals.sum()}")

# Display up to five mismatches with their differences.
car_clean.loc[mismatched_totals].assign(
    Category_Sum=category_totals[mismatched_totals],
    Difference=total_difference[mismatched_totals]
).head()

Rows where categories do not match the total: 0


,LSOA_Name,LSOA_Code,Total_Households,No_Car_Van_Households,One_Car_Van_Households,Two_Cars_Vans_Households,Three_Plus_Cars_Vans_Households,Category_Sum,Difference


### Findings

- The four car or van availability categories sum to Total_Households
  for all 203 LSOAs.
- No mismatches were found, so the displayed mismatch table is empty.
- No adjustments to the household counts are required.

---

## 12. Save the Cleaned Dataset

Save the validated dataset as a separate CSV without the dataframe index.
Keep the raw file unchanged.

In [16]:
# Developed with ChatGPT assistance to save the cleaned dataset.

# Define the cleaned output path.
output_path = "cleaned_datasets/dudley_car_availability_cleaned.csv"

# Save the cleaned data without an additional index column.
car_clean.to_csv(output_path, index=False)

# Report the saved file path and dataset dimensions.
print(f"Saved: {output_path}")
print(f"Rows: {len(car_clean)}")
print(f"Columns: {car_clean.shape[1]}")

Saved: cleaned_datasets/dudley_car_availability_cleaned.csv
Rows: 203
Columns: 7


### Findings

- The cleaned CSV was saved to
  `cleaned_datasets/dudley_car_availability_cleaned.csv`.
- The saved dataset contains 203 rows and 7 columns, without an
  additional index column.

---

## Conclusion

The Census 2021 car or van availability dataset is cleaned and validated.

Eight non-data rows were removed, columns were renamed, and household
counts were converted to integers without changing their values.
No missing values, duplicate records or duplicate LSOA codes remain.
All four availability categories sum correctly to the household total
for every LSOA.

The dataset covers the same 203 LSOAs as the cleaned economic activity
dataset and is ready for later analysis using LSOA_Code.

These figures describe households, not individuals. Having no car or
van does not establish that someone cannot travel to work; other
transport options must be considered when interpreting findings.
The raw CSV remains unchanged.